# Beam Noise Diffusion (BND) Experiment

This notebook clones the latest code from GitHub into the Colab runtime. Google Drive is used only for persistent runtime artifacts: Hugging Face cache, generated images, result JSON files, and CSV summaries.


## Step 0: Check GPU

Verify which GPU Colab assigned before loading the diffusion model.


In [8]:
!nvidia-smi

## Step 1: Mount Google Drive

Drive is used only as persistent storage for `cache/` and `outputs/`.


In [9]:
from google.colab import drive
drive.mount('/content/drive')

## Step 2: Clone Code from GitHub

The repository is cloned into the temporary Colab runtime. This keeps executable code separate from persistent outputs.


In [10]:
from pathlib import Path
import shutil
import subprocess
import sys

REPO_URL = "https://github.com/nguyenphamngocquy/beam-noise-diffusion.git"
CODE_ROOT = Path("/content/beam-noise-diffusion")
STORAGE_ROOT = Path("/content/drive/MyDrive/beam-noise-diffusion-storage")

if CODE_ROOT.exists():
    shutil.rmtree(CODE_ROOT)

subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(CODE_ROOT)], check=True)
sys.path.insert(0, str(CODE_ROOT))

print("Code root:", CODE_ROOT)
print("Storage root:", STORAGE_ROOT)

## Step 3: Install Dependencies

Install dependencies from the cloned repository's `requirements.txt`.


In [11]:
import subprocess
import sys

subprocess.run([
    sys.executable,
    "-m",
    "pip",
    "install",
    "-q",
    "-r",
    str(CODE_ROOT / "requirements.txt"),
], check=True)

## Step 4: Load Config and Prompts

Configs and prompts are read from the cloned repository. Runtime outputs are written to `STORAGE_ROOT` on Drive.


In [12]:
from bnd.config import load_config, load_prompts
from bnd.io import ensure_experiment_dirs

CONFIG_NAME = "bnd"
ensure_experiment_dirs(STORAGE_ROOT)
CFG = load_config(CODE_ROOT, CONFIG_NAME)
PROMPTS = load_prompts(CODE_ROOT)
RUN_NAME = CFG["name"]

print("Config:", RUN_NAME)
print("Number of prompts:", len(PROMPTS))
print(PROMPTS[0])

In [13]:
# Load HF_TOKEN from user input
import os
from getpass import getpass

token = getpass("Enter HF_TOKEN: ").strip()
os.environ["HF_TOKEN"] = token

print("HF_TOKEN set:", bool(os.environ.get("HF_TOKEN")))

In [ ]:
# Verify HF_TOKEN by calling whoami
from huggingface_hub import whoami

print(whoami(token=os.environ["HF_TOKEN"]))

## Step 5: Load Models

Stable Diffusion and CLIP are loaded from code in the cloned repo. Model files are cached on Drive under `STORAGE_ROOT/cache/hf_cache`.


In [14]:
from bnd.sampling import get_gpu_name, load_sd_pipeline
from bnd.scoring import ClipScorer

print("GPU:", get_gpu_name())
CACHE_DIR = STORAGE_ROOT / "cache" / "hf_cache"
pipe = load_sd_pipeline(CFG, CACHE_DIR)
scorer = ClipScorer(CACHE_DIR)

## Step 6: Run Experiments

For each prompt, the notebook runs random baseline, one optional Best-of-N baseline, Monte-Carlo selection, and BND. Set `PROMPT_LIMIT` to a small number for a smoke test.


In [15]:
from bnd.algorithm import run_best_of_n, run_bnd, run_monte_carlo, run_random_baseline

PROMPT_LIMIT = None  # Set to a small number, e.g. 2, for a quick smoke test
BEST_OF_N = 8  # Choose one value: 4, 8, 16, ...; set None to skip this baseline
items = PROMPTS if PROMPT_LIMIT is None else PROMPTS[:PROMPT_LIMIT]

for item in items:
    run_random_baseline(item, CFG, pipe, scorer, STORAGE_ROOT, RUN_NAME)
    if BEST_OF_N is not None:
        run_best_of_n(item, CFG, pipe, scorer, STORAGE_ROOT, RUN_NAME, n=BEST_OF_N)
    run_monte_carlo(item, CFG, pipe, scorer, STORAGE_ROOT, RUN_NAME)
    run_bnd(item, CFG, pipe, scorer, STORAGE_ROOT, RUN_NAME)


## Step 7: Summarize Results

Collect JSON result files from Drive into CSV summaries and display aggregate metrics.


In [16]:
from bnd.io import summarize_results

METHODS = ["random"]
if BEST_OF_N is not None:
    METHODS.append(f"best_of_{BEST_OF_N}")
METHODS += ["monte_carlo", "bnd"]

df, summary, group_summary = summarize_results(STORAGE_ROOT, RUN_NAME, methods=METHODS)
display(df.head())
display(summary)
display(group_summary)


In [17]:
try:
    from google.colab import runtime
    runtime.unassign()
except ImportError:
    print("Not running in Google Colab; skipping runtime disconnect.")